# API

This page considers the internal API of the airflow.

## DAG

A DAG represents a set of nodes (tasks) that are related to each other by the sequence of the execution within the graph. In the ariflow it represented by the `airflow.DAG` class.

---

The following cell defines the simple DAG:

In [7]:
from datetime import datetime

from airflow import DAG
from airflow.providers.standard.operators.empty import EmptyOperator


with DAG(
    dag_id="learning_dag",
    start_date=datetime(2024, 1, 1),
    schedule=None,
    catchup=False,
) as dag:

    start = EmptyOperator(task_id="start")

    middle = EmptyOperator(task_id="middle")

    end = EmptyOperator(task_id="end")

    start >> middle >> end

As the result the target of the context manager is the instance of the DAG:

In [8]:
dag

<DAG: learning_dag>

The tasks of the dag:

In [9]:
dag.tasks

[<Task(EmptyOperator): start>,
 <Task(EmptyOperator): middle>,
 <Task(EmptyOperator): end>]

The tasks in the topological order:

In [10]:
dag.topological_sort()

(<Task(EmptyOperator): start>,
 <Task(EmptyOperator): middle>,
 <Task(EmptyOperator): end>)

The starting nodes of the graph (`roots`) and finishing nodes (`leaves`).

In [11]:
dag.roots, dag.leaves

([<Task(EmptyOperator): start>], [<Task(EmptyOperator): end>])

## DagBag

The `DagBag` object in Airflow is responsible for loading DAGs. You can specify in which folder where your DAGs are located; Airflow will the "import" them and construct a `Dag` object for each.

---

The following cell creates a folder containing a simple Dag that will be loaded by the `DagBag` object.

In [3]:
!mkdir /tmp/experimental_dags

In [7]:
%%writefile /tmp/experimental_dags/simple_dag.py
from datetime import datetime

from airflow import DAG
from airflow.providers.standard.operators.empty import EmptyOperator

with DAG(
    dag_id="simple_dag",
    start_date=datetime(2024, 1, 1),
    schedule=None,
    catchup=False,
) as dag:
    task = EmptyOperator(
        task_id="task",
    )

Overwriting /tmp/experimental_dags/simple_dag.py


The process of the DAGs loading:

In [12]:
from airflow.dag_processing.dagbag import DagBag
dagbag = DagBag("/tmp/experimental_dags", include_examples=False)

2026-05-27T20:05:13.302650Z [info     ] Filling up the DagBag from /tmp/experimental_dags [airflow.dag_processing.dagbag.DagBag] loc=dagbag.py:469


{'simple_dag': <DAG: simple_dag>}

The `dags` attribute is a dictionary tha maps to the dags ids actuall dags instances.

In [13]:
dagbag.dags

{'simple_dag': <DAG: simple_dag>}